# Pure SPDNet on xDAWN-treated LWF_SHrinked Covariance Matrices

In [3]:
# Lib
import sys
sys.path.append("/home/rffl/files/ic/utils/")

import torch
import datagen, eegdataset #type: ignore
from spd_learn.models import SPDNet
from torch.utils.data import DataLoader
from torchmetrics.classification import ConfusionMatrix
import matplotlib.pyplot as plt
import numpy as np
import pyriemann as rmn

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
# Data config
classVec = [i+1 for i in range(10)]
subVec = [i+1 for i in range(10)]

nClass = len(classVec)
nSub = len(subVec)

In [ ]:
train_path, test_path = datagen.generatedataset(classVec, subVec, 0.75, False)
X_train = np.load(train_path, allow_pickle=True)
X_test = np.load(test_path, allow_pickle=True)

# generate target
y_train = np.array([])
part = len(X_train) // len(classVec)
for i in range(len(classVec)):
    y_train = np.concatenate([y_train, i * np.ones(part, dtype=np.int64)])

y_test = np.array([])
part = len(X_test) // len(classVec)
for i in range(len(classVec)):
    y_test = np.concatenate([y_test, i * np.ones(part, dtype=np.int64)])

# xDawn
xdawn = rmn.estimation.Xdawn(nfilter=5, estimator="lwf")
X_train = xdawn.fit_transform(X_train, y_train)
X_test = xdawn.transform(X_test)
np.save(train_path, X_train )
np.save(test_path, X_test )

trainDataset = eegdataset.KClassDataset(nClass, "./train.npy")
trainLoader = DataLoader(trainDataset, 40, True, num_workers=4)

testDataset = eegdataset.KClassDataset(nClass, "./test.npy")
testLoader = DataLoader(testDataset, 40, True, num_workers=4)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Saved ./train.npy shape = (6000, 17, 100)
Saved ./test.npy shape = (2000, 17, 100)
X_train shape: (6000, 100, 100)
X_test shape: (2000, 100, 100)


In [8]:
import time
def regular_train(model, classVec, trainLoader, testLoader, epochs, device):
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_hist_loss = []
    test_hist_loss = []

    train_hist_acc = []
    test_hist_acc = []

    begin = time.time()
    print(f"Begin {begin}")
    for epoch in range(epochs):
        train_loss, train_acc = train(model, trainLoader, criterion, optimizer, device)
        test_loss, test_acc = test(model, testLoader, criterion, device)
        train_hist_loss.append(train_loss); train_hist_acc.append(train_acc)
        test_hist_loss.append(test_loss); test_hist_acc.append(test_acc)
        print(f"Epoch {epoch+1}: "
        f"Train loss {train_loss:.4f}, acc {train_acc:.4f} | "
        f"Test loss {test_loss:.4f}, acc {test_acc:.4f} | "
        f"Time {time.time() - begin} s")

    confmat = ConfusionMatrix(task="multiclass", num_classes=len(classVec))    

    fig, axs = plt.subplots(1,2)
    axs[0].plot(range(epochs), train_hist_loss, label="Train Loss")
    axs[0].plot(range(epochs), test_hist_loss, label="Test Loss")
    axs[0].legend()

    axs[1].plot(range(epochs), train_hist_acc, label="Train Accuracy")
    axs[1].plot(range(epochs), test_hist_acc, label="Test Accuracy")
    axs[1].legend()
    fig.show()

    for X, y in testLoader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        preds = torch.argmax(logits, dim=1)

        confmat.update(preds.cpu(), y.cpu())

    confmatrix = confmat.compute()
    plot_confusion_matrix(confmatrix, classVec)

def train(model, trainLoader, criterion, optimizer, device):
    torch.cuda.empty_cache()
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in trainLoader:
        X = X.to(device); y = y.to(device)

        optimizer.zero_grad()
        y_hat = model(X)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = y_hat.argmax(dim=1)
        correct += pred.eq(y).sum().item()
        total += y.size(0)
    
    avg_loss = total_loss / len(trainLoader)
    acc = correct / total
    return avg_loss, acc

@torch.no_grad()
def test(model, testLoader, criterion, device):
    torch.cuda.empty_cache()
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in testLoader:
        X = X.to(device); y = y.to(device)

        y_hat = model(X)
        total_loss += criterion(y_hat, y).item()

        pred = y_hat.argmax(dim=1)
        correct += pred.eq(y).sum().item()
        total += y.size(0)
    
    avg_loss = total_loss / len(testLoader)
    acc = correct / total
    return avg_loss, acc

def plot_confusion_matrix(confmatrix, class_names=None):
    plt.figure(figsize=(6, 5))
    plt.imshow(confmatrix, interpolation='nearest')
    plt.title("Confusion Matrix")
    plt.colorbar()

    num_classes = confmatrix.shape[0]

    # Tick labels
    if not class_names:
        class_names = [str(i) for i in range(num_classes)]

    plt.xticks(range(num_classes), class_names, rotation=45)
    plt.yticks(range(num_classes), class_names)

    # Print values inside cells
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, str(confmatrix[i, j].item()),
                     ha="center", va="center", color="white" if confmatrix[i, j] > confmatrix.max()/2 else "black")

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

In [13]:
device = torch.device("cuda:1")
model = SPDNet("cov", n_chans=100, n_outputs=10).to(device)
regular_train(model, classVec, trainLoader, testLoader, 30, device)

Begin 1780589121.7743402


/home/rffl/miniconda3/envs/eeg/lib/python3.11/site-packages/spd_learn/models/spdnet.py:70: UserWarning: subspacedim is None, using the default value of the number of channels
  warn(


Epoch 1: Train loss 1.8965, acc 0.3360 | Test loss 1.5992, acc 0.4495 | Time 32.16304326057434 s
Epoch 2: Train loss 1.1189, acc 0.6392 | Test loss 1.6574, acc 0.4450 | Time 62.09514832496643 s
Epoch 3: Train loss 0.8259, acc 0.7400 | Test loss 1.6725, acc 0.4530 | Time 92.15361714363098 s
Epoch 4: Train loss 0.6761, acc 0.8033 | Test loss 1.7218, acc 0.4560 | Time 122.44297552108765 s
Epoch 5: Train loss 0.5802, acc 0.8378 | Test loss 1.7556, acc 0.4605 | Time 152.83015894889832 s
Epoch 6: Train loss 0.4889, acc 0.8667 | Test loss 1.8327, acc 0.4500 | Time 183.25453639030457 s
Epoch 7: Train loss 0.4476, acc 0.8777 | Test loss 1.9601, acc 0.4390 | Time 213.7909426689148 s
Epoch 8: Train loss 0.3924, acc 0.8948 | Test loss 1.9647, acc 0.4550 | Time 244.54240798950195 s
Epoch 9: Train loss 0.3548, acc 0.9092 | Test loss 1.9937, acc 0.4510 | Time 275.23249554634094 s
Epoch 10: Train loss 0.3136, acc 0.9238 | Test loss 2.0813, acc 0.4510 | Time 306.00123262405396 s
Epoch 11: Train loss 0.

KeyboardInterrupt: 